Interpolacion Suave

In [ ]:
import cv2
import numpy as np
import supervision as sv
from inference import get_model

def load_model():
    return get_model(model_id="football-ball-detection-rejhg/4", api_key='ncdFToGYVEqV6Ll2cmj3')

def initialize_video(video_file):
    video_capture = cv2.VideoCapture(video_file)
    frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(video_capture.get(cv2.CAP_PROP_FPS))
    return video_capture, frame_width, frame_height, fps

def create_video_writer(output_file, frame_width, frame_height, fps):
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    return cv2.VideoWriter(output_file, fourcc, fps, (frame_width, frame_height))

def process_video(video_capture, video_writer, model):
    last_center = None
    while video_capture.isOpened():
        ret, frame = video_capture.read()
        if not ret:
            break

        results = model.infer(frame)[0]
        detections = sv.Detections.from_inference(results)

        if len(detections) > 0:
            detection = detections[0]
            x1, y1, x2, y2 = detection.xyxy[0]
            center = (int((x1 + x2) / 2), int((y1 + y2) / 2))
            last_center = center
        else:
            center = last_center

        if center is not None:
            draw_triangle(frame, center)

        video_writer.write(frame)

def draw_triangle(frame, center):
    offset_y = 30
    points = np.array([
        [center[0], center[1] + 20 - offset_y],
        [center[0] - 10, center[1] - 10 - offset_y],
        [center[0] + 10, center[1] - 10 - offset_y]
    ])
    cv2.drawContours(frame, [points], 0, (0, 255, 0), -1)

def main():
    video_file = "football.mp4"
    output_file = "salida.avi"

    model = load_model()
    video_capture, frame_width, frame_height, fps = initialize_video(video_file)
    video_writer = create_video_writer(output_file, frame_width, frame_height, fps)

    process_video(video_capture, video_writer, model)

    video_capture.release()
    video_writer.release()

if __name__ == "__main__":
    main()


Otro

In [ ]:
from inference import get_model
import supervision as sv
import cv2
import numpy as np
video_file = "football.mp4"
output_file = "salida_para_fer.avi"

model = get_model(model_id="football-ball-detection-rejhg/4", api_key='ncdFToGYVEqV6Ll2cmj3')


video_capture = cv2.VideoCapture(video_file)


frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(video_capture.get(cv2.CAP_PROP_FPS))


fourcc = cv2.VideoWriter_fourcc(*'XVID')
video_writer = cv2.VideoWriter(output_file, fourcc, fps, (frame_width, frame_height))


prev_center = None


while video_capture.isOpened():
    ret, frame = video_capture.read()
    if not ret:
        break
    
    
    results = model.infer(frame)[0]
    
    
    detections = sv.Detections.from_inference(results)
    
    
    if len(detections) > 0:
        detection = detections[0]  
        x1, y1, x2, y2 = detection.xyxy[0]
        center = (
            int((x1 + x2) / 2),
            int((y1 + y2) / 2)
        )
    else:
        center = None
    if prev_center is not None and center is not None:
        interpolated_center = (
            int((prev_center[0] + center[0]) / 2),
            int((prev_center[1] + center[1]) / 2)
        )
    else:
        interpolated_center = center
    if interpolated_center is not None:
        points = np.array([
            [interpolated_center[0], interpolated_center[1] + 10-30],
            [interpolated_center[0] - 10, interpolated_center[1] - 10-30],  
            [interpolated_center[0] + 10, interpolated_center[1] - 10-30]   
        ])
        cv2.drawContours(frame, [points], 0, (0, 255, 0), -1)  
    video_writer.write(frame)
    prev_center = center
video_capture.release()
video_writer.release()
